In [1]:
# ============================================================
# preprocess_stream_stead.py
# Preprocessing STEAD 3-Komponen (Streaming, SSD, Progress Bar)
# ============================================================

import numpy as np
import pandas as pd
import h5py
import ipywidgets
ipywidgets.IntProgress()

from scipy.signal import butter, filtfilt
from tqdm.notebook import tqdm   # penting untuk Jupyter!

# ------------------------------------------------------------
# Path data STEAD di SSD eksternal
# ------------------------------------------------------------
CSV_PATH = "//Volumes/Local Disk/Code_Git/S3_code/seismic/data/merge.csv"
H5_PATH  = "//Volumes/Local Disk/Code_Git/S3_code/seismic/data/merge.hdf5"
OUT_PATH = "/Volumes/Extreme SSD/stream_stead/output/stead_processed.npy"

'/Volumes/Local Disk/Code_Git/S3_code/seismic/data'

# ------------------------------------------------------------
# 1. Bandpass filter 1–20 Hz
# ------------------------------------------------------------
def bandpass_filter(data, fs=100, low=1.0, high=20.0, order=4):
    nyq = 0.5 * fs
    b, a = butter(order, [low/nyq, high/nyq], btype='band')
    return filtfilt(b, a, data)

# ------------------------------------------------------------
# 2. Normalisasi per-komponen (z-score)
# ------------------------------------------------------------
def normalize_trace(x):
    mean = np.mean(x)
    std = np.std(x)
    return (x - mean) / std if std > 1e-6 else (x - mean)

# ------------------------------------------------------------
# 3. Padding atau trimming ke 6000 sampel
# ------------------------------------------------------------
def fix_length(x, target_len=6000):
    if len(x) > target_len:
        return x[:target_len]
    elif len(x) < target_len:
        return np.pad(x, (0, target_len - len(x)), mode='constant')
    return x

# ------------------------------------------------------------
# 4. Preprocessing lengkap untuk satu event STEAD
# ------------------------------------------------------------
def preprocess_event(w):
    Z = fix_length(normalize_trace(bandpass_filter(w[:, 0])))
    N = fix_length(normalize_trace(bandpass_filter(w[:, 1])))
    E = fix_length(normalize_trace(bandpass_filter(w[:, 2])))
    return np.stack([Z, N, E], axis=-1)

# ------------------------------------------------------------
# 5. Preprocessing streaming aman untuk dataset besar
# ------------------------------------------------------------
def run_preprocessing_streaming():
    print("[STEP] Membaca metadata STEAD...")
    df = pd.read_csv(CSV_PATH)
    n = len(df)
    print(f"[INFO] Total event: {n}")

    print("[STEP] Menyiapkan file output memmap di SSD...")
    out = np.memmap(OUT_PATH, dtype='float32', mode='w+', shape=(n, 6000, 3))

    print("[STEP] Membuka file HDF5 dan memulai streaming...")

    with h5py.File(H5_PATH, "r") as f:
        data = f["data"]

        # Progress bar Jupyter yang pasti muncul
        for i, name in enumerate(
            tqdm(df.trace_name.values, desc="Preprocessing STEAD (Streaming)", mininterval=0.1)
        ):
            if name in data:
                w = data[name][:]
                out[i] = preprocess_event(w)
            else:
                out[i] = np.zeros((6000, 3), dtype='float32')

    print("\n[DONE] Preprocessing selesai.")
    print(f"[INFO] Hasil disimpan ke: {OUT_PATH}")

# ------------------------------------------------------------
# 6. Jalankan jika file ini dieksekusi langsung
# ------------------------------------------------------------
if __name__ == "__main__":
    run_preprocessing_streaming()


[STEP] Membaca metadata STEAD...


/var/folders/lt/2mkl6ry53ll9fdk2br6skfgw0000gn/T/ipykernel_79210/196708232.py:64: DtypeWarning: Columns (7,11,13,14,24,25,26,30,31) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(CSV_PATH)


[INFO] Total event: 1265657
[STEP] Menyiapkan file output memmap di SSD...
[STEP] Membuka file HDF5 dan memulai streaming...


Preprocessing STEAD (Streaming):   0%|          | 0/1265657 [00:00<?, ?it/s]


[DONE] Preprocessing selesai.
[INFO] Hasil disimpan ke: /Volumes/Extreme SSD/stream_stead/output/stead_processed.npy
